# 1 Stage Try

This notebook trains a direct one-stage model that maps noisy gradient data straight to masks for `N=8`. It uses the same best decoder family as the current two-stage baseline for a fair comparison. Outputs are written to `outputs/one_stage_try/`.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

ROOT = Path.cwd()
while not (ROOT / 'src').exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))

from config import Stage2ModelConfig, StageTrainingConfig, TwoStageRunConfig, TwoStageStackConfig
from datasets import build_two_stage_datasets, save_two_stage_dataset
from models import Stage2CoordConvDecoder, evaluate_stage2_predictions, evaluate_stage2_predictions_by_shape, fit_stage2_model, predict_stage2_logits, select_best_stage2_threshold, set_torch_seed
from plotting import save_measurement_points_plot, save_reconstruction_examples, save_shape_gallery
from shapes import create_fixed_benchmark_shapes

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED = 42
np.random.seed(SEED)
set_torch_seed(SEED)

OUTPUT_ROOT = ROOT / 'outputs' / 'one_stage_try'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

run_config = TwoStageRunConfig(
    N=8,
    training_samples=10000,
    validation_samples=2000,
    test_samples=500,
    rho=0.8,
    grid_size=32,
    threshold=0.5,
    use_validation_threshold_sweep=True,
    noise_level=0.01,
    seed=SEED,
    training_shape_weights=(
        ('rectangle', 0.30),
        ('two_circles', 0.30),
        ('annulus', 0.15),
        ('ellipse', 0.15),
        ('circle', 0.10),
    ),
    model=TwoStageStackConfig(
        stage2=Stage2ModelConfig(
            hidden_layer_sizes=(512, 1024),
            dropout_rates=(0.10, 0.10),
            model_type='coord_conv_decoder',
            latent_grid_size=16,
            latent_channels=160,
            decoder_channels=(160, 128, 96, 64, 32),
            use_rectangle_edge_weighting=True,
            use_foreground_pos_weight=False,
            rectangle_edge_weight=4.0,
            rectangle_edge_width=3,
            edge_weight_mode='rectangle',
            annulus_edge_weight=1.0,
            annulus_edge_width=3,
            training=StageTrainingConfig(
                epochs=170,
                batch_size=96,
                learning_rate=0.0005,
                validation_frequency=60,
                verbose=False,
                early_stopping_patience=24,
                min_epochs=50,
                min_improvement=0.002,
                lr_drop_factor=0.5,
                lr_drop_period=80,
                weight_decay=0.00025,
                gradient_clip_norm=0.8,
                loss_type='bce_dice',
                dice_loss_weight=1.0,
                dice_smooth=1.0,
            ),
        ),
    ),
    output_dir=OUTPUT_ROOT,
)

run_output_dir = run_config.run_output_dir
run_output_dir.mkdir(parents=True, exist_ok=True)
dataset_bundle = build_two_stage_datasets(run_config)
dataset_paths = save_two_stage_dataset(dataset_bundle, run_output_dir / 'datasets')

model = Stage2CoordConvDecoder(
    input_dim=run_config.gradient_feature_size,
    output_dim=run_config.mask_pixels,
    hidden_dims=run_config.model.stage2.hidden_layer_sizes,
    dropout_rates=run_config.model.stage2.dropout_rates,
    latent_grid_size=run_config.model.stage2.latent_grid_size,
    latent_channels=run_config.model.stage2.latent_channels,
    decoder_channels=run_config.model.stage2.decoder_channels,
)

training = run_config.model.stage2.training
training_result = fit_stage2_model(
    model=model,
    train_features=dataset_bundle.train.gradient_data,
    train_targets=dataset_bundle.train.masks,
    val_features=dataset_bundle.validation.gradient_data,
    val_targets=dataset_bundle.validation.masks,
    epochs=training.epochs,
    batch_size=training.batch_size,
    learning_rate=training.learning_rate,
    device=DEVICE,
    validation_frequency=training.validation_frequency,
    verbose=training.verbose,
    early_stopping_patience=training.early_stopping_patience,
    train_shape_types=dataset_bundle.train.shape_types,
    grid_size=run_config.grid_size,
    use_rectangle_edge_weighting=run_config.model.stage2.use_rectangle_edge_weighting,
    rectangle_edge_weight=run_config.model.stage2.rectangle_edge_weight,
    rectangle_edge_width=run_config.model.stage2.rectangle_edge_width,
    edge_weight_mode=run_config.model.stage2.edge_weight_mode,
    annulus_edge_weight=run_config.model.stage2.annulus_edge_weight,
    annulus_edge_width=run_config.model.stage2.annulus_edge_width,
    min_epochs=training.min_epochs,
    min_improvement=training.min_improvement,
    lr_drop_factor=training.lr_drop_factor,
    lr_drop_period=training.lr_drop_period,
    weight_decay=training.weight_decay,
    gradient_clip_norm=training.gradient_clip_norm,
    loss_type=training.loss_type,
    dice_loss_weight=training.dice_loss_weight,
    dice_smooth=training.dice_smooth,
    use_foreground_pos_weight=run_config.model.stage2.use_foreground_pos_weight,
)

predicted_validation_masks = predict_stage2_logits(model, dataset_bundle.validation.gradient_data, device=DEVICE, training_result=training_result)
predicted_test_masks = predict_stage2_logits(model, dataset_bundle.test.gradient_data, device=DEVICE, training_result=training_result)
predicted_fixed_masks = predict_stage2_logits(model, dataset_bundle.fixed.gradient_data, device=DEVICE, training_result=training_result)

threshold_summary = select_best_stage2_threshold(
    dataset_bundle.validation.masks,
    predicted_validation_masks,
    run_config.threshold_candidates,
)
threshold_summary['selection_mode'] = 'validation_sweep'
threshold_used = float(threshold_summary['selected_threshold'])

metrics = {
    'stage2_test': evaluate_stage2_predictions(dataset_bundle.test.masks, predicted_test_masks, threshold_used),
    'stage2_fixed': evaluate_stage2_predictions(dataset_bundle.fixed.masks, predicted_fixed_masks, threshold_used),
}
metrics_by_shape = {
    'stage2_test': evaluate_stage2_predictions_by_shape(dataset_bundle.test.masks, predicted_test_masks, threshold_used, dataset_bundle.test.shape_types),
    'stage2_fixed': evaluate_stage2_predictions_by_shape(dataset_bundle.fixed.masks, predicted_fixed_masks, threshold_used, dataset_bundle.fixed.shape_types),
}

save_measurement_points_plot(dataset_bundle.measurement_points, run_output_dir / 'measurement_points.png')
save_shape_gallery(create_fixed_benchmark_shapes(), run_output_dir / 'fixed_shapes.png', grid_size=run_config.grid_size)
save_reconstruction_examples(dataset_bundle.fixed.masks, predicted_fixed_masks, dataset_bundle.fixed.names, run_output_dir / 'fixed_reconstructions.png', run_config.grid_size, threshold_used)
torch.save(model.state_dict(), run_output_dir / 'stage2_model.pt')

def plot_history(history: dict, title: str, output_path: Path):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(np.arange(1, len(history['train_loss']) + 1), history['train_loss'], marker='o')
    axes[0].set_title(f'{title} Train Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].grid(True, alpha=0.3)
    axes[1].plot(history['validation_steps'], history['val_loss'], marker='o', color='tab:orange')
    axes[1].set_title(f'{title} Validation Loss')
    axes[1].set_xlabel('Validation Step')
    axes[1].set_ylabel('Loss')
    axes[1].grid(True, alpha=0.3)
    fig.tight_layout()
    fig.savefig(output_path, dpi=150, bbox_inches='tight')
    plt.close(fig)

plot_history(training_result.history, 'One-Stage Model', run_output_dir / 'training_losses.png')

summary = {
    'config': {
        'N': run_config.N,
        'training_samples': run_config.training_samples,
        'validation_samples': run_config.validation_samples,
        'test_samples': run_config.test_samples,
        'threshold': threshold_used,
        'training_shape_weights': list(run_config.training_shape_weights),
        'model_type': run_config.model.stage2.model_type,
        'latent_grid_size': run_config.model.stage2.latent_grid_size,
        'latent_channels': run_config.model.stage2.latent_channels,
        'decoder_channels': list(run_config.model.stage2.decoder_channels),
    },
    'condition_number': dataset_bundle.matrix_condition_number,
    'metrics': metrics,
    'metrics_by_shape': metrics_by_shape,
    'threshold_summary': threshold_summary,
    'dataset_paths': {key: str(value) for key, value in dataset_paths.items()},
    'training_history': training_result.history,
}

with (run_output_dir / 'summary.json').open('w', encoding='utf-8') as handle:
    json.dump(summary, handle, indent=2)
with (OUTPUT_ROOT / 'sweep_summary.json').open('w', encoding='utf-8') as handle:
    json.dump({'runs': [summary]}, handle, indent=2)

print('One-stage test IoU:', metrics['stage2_test']['mean_iou'])
print('One-stage fixed IoU:', metrics['stage2_fixed']['mean_iou'])
print('Selected threshold:', threshold_used)
